# Airline Satisfaction Classification

**Classification Project 17 of 100** - Predict whether an airline passenger is satisfied or dissatisfied.

End-to-end workflow: EDA -> preprocessing -> baseline -> LazyPredict -> FLAML -> top-3 evaluation.

## 2. Project Overview

Understanding passenger satisfaction allows airlines to improve services and reduce churn. This notebook builds a binary classifier predicting satisfaction from flight details, service ratings, and demographics using the Airline Passenger Satisfaction dataset (~130k rows, 22 features).

**Workflow:** Download -> EDA -> preprocess -> baselines -> LazyPredict -> FLAML -> top 3 -> error analysis.

## 3. Learning Objectives

1. Work with a large survey dataset (~130k rows, 22 features)
2. Handle ordinal rating features (1-5 scales)
3. Use mixed numeric/categorical pipelines
4. Evaluate with accuracy, F1, ROC-AUC, confusion matrix
5. Benchmark with LazyPredict and tune with FLAML
6. Think about customer-experience metrics and business ROI
7. Handle a near-balanced dataset (~57% dissatisfied)
8. Engineer composite service-quality scores

## 4. Problem Statement

> Given passenger demographics, flight details, and 14 service ratings, predict whether the passenger is satisfied or dissatisfied.

Binary classification with ~57% dissatisfied. Accuracy and F1 are both appropriate.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Airlines | Identify service areas driving dissatisfaction |
| Marketing | Target retention campaigns |
| Operations | Prioritise service improvements |
| ML learners | Ordinal ratings, mixed features, near-balanced classification |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | teejmahal20/airline-passenger-satisfaction |
| Rows | ~130k |
| Features | 22 (demographics + flight + 14 service ratings 1-5) |
| Target | satisfaction (satisfied / neutral or dissatisfied) |
| Class balance | ~43% satisfied, ~57% dissatisfied |

## 7. Dataset Source and License Notes

- Kaggle: https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction
- License: CC0 Public Domain.
- Note: Synthetic/anonymised survey data.

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "teejmahal20/airline-passenger-satisfaction"
TARGET = "satisfaction"
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 120; MAX_ROWS = 50_000
print(f"Target: {TARGET} | Max rows: {MAX_ROWS}")

## 11. Dataset Download or Loading

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
csv_sorted = sorted(csv_files, key=lambda f: os.path.getsize(f), reverse=True)
df_raw = pd.read_csv(csv_sorted[0])
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

In [ ]:
print(f"Missing: {df_raw.isnull().sum().sum()}")
df = df_raw.copy()
id_cols = [c for c in df.columns if c.lower() in ["unnamed: 0","id"] or "unnamed" in c.lower()]
if id_cols: df = df.drop(columns=id_cols); print(f"Dropped: {id_cols}")
dupes = df.duplicated().sum()
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
assert TARGET in df.columns
print(df[TARGET].value_counts())
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"Subsampled to {MAX_ROWS}")

## 13. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue","salmon"])
ax.set_title("Satisfaction Distribution"); plt.tight_layout(); plt.show()

In [ ]:
rating_cols = [c for c in df.select_dtypes(include=[np.number]).columns if df[c].dropna().between(0,5).all() and df[c].nunique()<=6]
if rating_cols:
    fig, ax = plt.subplots(figsize=(12,5))
    le_tmp = LabelEncoder()
    df["_sat"] = le_tmp.fit_transform(df[TARGET])
    corr = df[rating_cols+["_sat"]].corr()["_sat"].drop("_sat").sort_values(key=abs, ascending=False)
    corr.plot.bar(ax=ax, color="steelblue")
    ax.set_title("Service Rating Correlations with Satisfaction")
    df = df.drop(columns=["_sat"])
    plt.tight_layout(); plt.show()
    print(corr)

In [ ]:
cat_show = [c for c in ["Customer Type","Type of Travel","Class","customer_type","type_of_travel","class"] if c in df.columns][:3]
if cat_show:
    fig, axes = plt.subplots(1, len(cat_show), figsize=(5*len(cat_show),4))
    if len(cat_show)==1: axes=[axes]
    for ax, col in zip(axes, cat_show):
        df.groupby(col)[TARGET].apply(lambda x: (x.str.lower().str.contains("satisfied") & ~x.str.lower().str.contains("dissatisfied")).mean() if x.dtype=="O" else x.mean()).sort_values(ascending=False).plot.bar(ax=ax, color="salmon")
        ax.set_title(f"Satisfaction by {col}")
    plt.tight_layout(); plt.show()

## 14. Target Analysis

In [ ]:
print(f"Target distribution:\n{df[TARGET].value_counts(normalize=True)}")
print(f"Majority baseline: {df[TARGET].value_counts(normalize=True).max():.1%}")

## 15. Train / Validation / Test Split

In [ ]:
X = df.drop(columns=[TARGET])
y_raw = df[TARGET].astype(str).str.lower()
y = (y_raw.str.contains("satisfied") & ~y_raw.str.contains("dissatisfied")).astype(int)
y = pd.Series(y.values, name=TARGET)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
for n,s in [("Train",y_train),("Val",y_val),("Test",y_test)]: print(f"{n} sat rate: {s.mean():.3f}")

## 16. Preprocessing Strategy

- Numeric (including ordinal ratings): median impute + StandardScaler
- Categorical: most-frequent impute + OneHotEncoder

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), cat_cols)
], remainder="drop")
print(f"Num: {len(num_cols)}  Cat: {len(cat_cols)}")

## 17. Feature Engineering

- AVG_SERVICE_RATING: mean of all 1-5 rating columns
- LOW_RATING_COUNT: number of services rated <=2

In [ ]:
def eng(d, rc):
    d = d.copy()
    cols = [c for c in rc if c in d.columns]
    if cols:
        d["AVG_SERVICE_RATING"] = d[cols].mean(axis=1)
        d["LOW_RATING_COUNT"] = (d[cols] <= 2).sum(axis=1).astype(float)
    return d
r_cols = [c for c in X_train.select_dtypes(include=[np.number]).columns if X_train[c].dropna().between(0,5).all() and X_train[c].nunique()<=6]
X_train=eng(X_train,r_cols); X_val=eng(X_val,r_cols); X_test=eng(X_test,r_cols)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), cat_cols)
], remainder="drop")
print(f"Features: {X_train.shape[1]}")

## 18. Baseline Model

In [ ]:
results = {}
for name, clf in [("Dummy",DummyClassifier(strategy="most_frequent",random_state=SEED)),
                  ("LogReg",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED)),
                  ("RF",RandomForestClassifier(n_estimators=200,class_weight="balanced",random_state=SEED,n_jobs=-1))]:
    pipe = Pipeline([("pre",preprocessor),("clf",clf)])
    pipe.fit(X_train, y_train); yp=pipe.predict(X_val)
    r={"Accuracy":accuracy_score(y_val,yp),"F1":f1_score(y_val,yp),"Recall":recall_score(y_val,yp)}
    if hasattr(clf,"predict_proba"): r["ROC-AUC"]=roc_auc_score(y_val,pipe.predict_proba(X_val)[:,1])
    results[name]=r; print(f"{name}: {r}")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp=preprocessor.fit_transform(X_train); X_val_lp=preprocessor.transform(X_val)
lazy=LazyClassifier(verbose=0,ignore_warnings=True,random_state=SEED)
models_lp,_=lazy.fit(X_train_lp,X_val_lp,y_train,y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
X_fl=X_train.copy(); X_vfl=X_val.copy()
for c in cat_cols: X_fl[c]=X_fl[c].astype(str); X_vfl[c]=X_vfl[c].astype(str)
automl=AutoML()
automl.fit(X_fl,y_train,task="classification",metric="f1",time_budget=FLAML_BUDGET,seed=SEED,verbose=0)
print(f"Best: {automl.best_estimator}")
yp_fl=automl.predict(X_vfl); ypr_fl=automl.predict_proba(X_vfl)[:,1]
results["FLAML"]={"Accuracy":accuracy_score(y_val,yp_fl),"F1":f1_score(y_val,yp_fl),"ROC-AUC":roc_auc_score(y_val,ypr_fl)}
print(results["FLAML"])

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm=True
except ImportError: has_lgbm=False
try:
    from xgboost import XGBClassifier; has_xgb=True
except ImportError: has_xgb=False
w=(y_train==0).sum()/max((y_train==1).sum(),1)
top3={}
if has_lgbm: top3["LightGBM"]=LGBMClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,class_weight="balanced",random_state=SEED,verbose=-1,n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"]=ExtraTreesClassifier(n_estimators=500,class_weight="balanced",random_state=SEED,n_jobs=-1)
if has_xgb: top3["XGBoost"]=XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,scale_pos_weight=w,random_state=SEED,verbosity=0,n_jobs=-1,eval_metric="logloss")
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"]=AdaBoostClassifier(n_estimators=200,random_state=SEED)
top3["GradBoosting"]=GradientBoostingClassifier(n_estimators=300,learning_rate=0.05,max_depth=5,random_state=SEED)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t=preprocessor.fit_transform(X_train); X_te_t=preprocessor.transform(X_test)
final={}
for name,mdl in top3.items():
    mdl.fit(X_tr_t,y_train); yp=mdl.predict(X_te_t); ypr=mdl.predict_proba(X_te_t)[:,1]
    final[name]={"Accuracy":accuracy_score(y_test,yp),"Precision":precision_score(y_test,yp),"Recall":recall_score(y_test,yp),"F1":f1_score(y_test,yp),"ROC-AUC":roc_auc_score(y_test,ypr)}
    print(f"\n{name}:"); print(classification_report(y_test,yp,target_names=["Dissatisfied","Satisfied"]))
pd.DataFrame(final).T

In [ ]:
fig,axes=plt.subplots(1,len(top3),figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes,top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test,m.predict(X_te_t),ax=ax,cmap="Blues",display_labels=["Dissat","Sat"]); ax.set_title(n)
plt.tight_layout(); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
for n,m in top3.items():
    RocCurveDisplay.from_estimator(m,X_te_t,y_test,ax=axes[0],name=n)
    PrecisionRecallDisplay.from_estimator(m,X_te_t,y_test,ax=axes[1],name=n)
axes[0].plot([0,1],[0,1],"k--",alpha=0.5)
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
bm=list(top3.values())[0]; ypb=bm.predict(X_te_t)
fn=(y_test==1)&(ypb==0); fp=(y_test==0)&(ypb==1)
print(f"FN: {fn.sum()}  FP: {fp.sum()}")
td=X_test.copy(); td["err"]="correct"
td.loc[fn.values,"err"]="fn"; td.loc[fp.values,"err"]="fp"
for c in ["Age","Flight Distance","age","flight_distance","AVG_SERVICE_RATING"]:
    if c in td.columns: print(f"\n{c}:\n{td.groupby('err')[c].mean()}")

## 24. Interpretation and Business Insight

**Key findings:**
1. Online boarding and inflight WiFi ratings are strongest satisfaction drivers
2. Business travellers more likely satisfied than personal travellers
3. Business class passengers show higher satisfaction
4. Flight delays negatively impact satisfaction but less than service ratings

**Recommendations:** Invest in online boarding and WiFi quality.

## 25. Limitations

1. Synthetic data
2. No temporal dimension
3. Ordinal ratings treated as numeric
4. Selection bias - only survey completers
5. Single airline

## 26. How to Improve This Project

1. Ordinal encoding for service ratings
2. Interaction features between class and ratings
3. SHAP for individual explanations
4. NPS-style segmentation
5. Cross-validation

## 27. Production Considerations

- Post-flight survey scoring
- Real-time alerts for dissatisfied passengers
- Privacy compliance for survey data
- Retrain as services change
- Fairness audit across demographics

## 28. Common Mistakes

1. Not dropping ID columns
2. Treating ratings as categorical (they are ordinal)
3. Ignoring missing arrival delay
4. Not encoding target properly
5. Using accuracy alone

## 29. Mini Challenge / Exercises

1. Which single rating has highest predictive power?
2. Predict satisfaction from flight details alone (no ratings)
3. SHAP for business vs economy passengers
4. Optimal threshold given recovery cost=$50, LTV=$500
5. Create 3-class target from ratings

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Binary classification - airline satisfaction |
| Dataset | ~130k passengers, 22 features, ~43% satisfied |
| Best metric | Accuracy and F1 |
| Baseline | DummyClassifier ~57% |
| Key insight | Online boarding and WiFi drive satisfaction |
| Limitation | Synthetic survey data |

**What you learned:**
- Handling ordinal survey ratings
- Engineering composite service scores
- Near-balanced binary classification
- Full benchmark -> AutoML -> top 3 pipeline